# 📓 Notebook 11: Fine-Tuning Basics
## GenAI Foundry — Part 2: Introductory Concepts

> **🔑 API Key Required** — This notebook uses the OpenAI API for fine-tuning. You will need an OpenAI API key.
> Estimated cost: **$1–$5** depending on the dataset size and training epochs.

---

## What You Will Learn

By the end of this notebook you will understand:

1. **What fine-tuning is** — and how it differs from prompt engineering and RAG
2. **When to fine-tune** — the decision framework: prompt first, RAG second, fine-tune third
3. **How to prepare training data** — the JSONL format OpenAI requires
4. **How to run a fine-tuning job** — using the OpenAI `fine_tuning.jobs` API
5. **How to evaluate a fine-tuned model** — comparing base vs. fine-tuned responses
6. **Cost and tradeoffs** — when fine-tuning is worth it

---

## The Mental Model: Three Ways to Teach a Model

```
┌─────────────────────────────────────────────────────────────────┐
│  APPROACH          ANALOGY              WHEN TO USE             │
│  ──────────────────────────────────────────────────────────── │
│  Prompt Engineering  Tell it what to do  Fast iteration, low cost│
│  RAG               Give it a textbook   Dynamic, changing docs  │
│  Fine-Tuning        Train it directly    Consistent style/format │
└─────────────────────────────────────────────────────────────────┘
```

Fine-tuning is NOT about teaching the model new facts — it is about shaping **how** it responds.
Use it when you need consistent tone, format, or domain-specific behaviour that prompts alone cannot reliably produce.

## Part 1: Setup

Install the OpenAI library and set your API key.

In [ ]:
# Install required library
!pip install openai --quiet

In [ ]:
import openai
import os
import json
import time

# Set your API key
# Option A: Set as environment variable (recommended)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# Option B: Enter directly (for Colab only — don't commit this to a repo!)
from getpass import getpass
api_key = getpass("Enter your OpenAI API key: ")
client = openai.OpenAI(api_key=api_key)

print("✅ OpenAI client configured")

## Part 2: Understanding the Training Data Format

OpenAI fine-tuning uses **JSONL** (JSON Lines) format. Each line is a complete conversation example with a system message, user message, and the expected assistant response.

```
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

**Key rules:**
- Minimum **10 examples** recommended (50–100 is better)
- Every example must have a `system`, `user`, and `assistant` message
- The `assistant` message is what the model learns to produce
- Quality matters far more than quantity

In [ ]:
# Build a small training dataset
# Scenario: We want a model that always responds in a concise, bullet-point style
# for a technical support use case.

SYSTEM_MSG = "You are a concise technical support assistant. Always respond using bullet points. Never use paragraphs. Be direct and actionable."

training_examples = [
    {
        "user": "My Docker container won't start.",
        "assistant": "- Run `docker logs <container_id>` to see the error\n- Check if the port is already in use: `lsof -i :8501`\n- Verify your Dockerfile has no syntax errors\n- Try `docker system prune` to clear stale resources"
    },
    {
        "user": "How do I reset my API key?",
        "assistant": "- Go to platform.openai.com\n- Click your profile → View API keys\n- Click 'Create new secret key'\n- Delete the old key once the new one works\n- Update your .env file with the new key"
    },
    {
        "user": "The app is running slowly.",
        "assistant": "- Check CPU/memory usage: `docker stats`\n- Look for large context windows in your prompts\n- Consider switching to a faster model (e.g., gpt-4o-mini)\n- Cache repeated API responses where possible\n- Profile which step is the bottleneck"
    },
    {
        "user": "I'm getting a rate limit error.",
        "assistant": "- Add exponential backoff: wait 1s, 2s, 4s between retries\n- Check your usage tier at platform.openai.com\n- Reduce request frequency or batch smaller inputs\n- Use `gpt-4o-mini` instead of `gpt-4o` to stay within limits\n- Consider upgrading your OpenAI usage tier"
    },
    {
        "user": "How do I install Python dependencies?",
        "assistant": "- Open a terminal in your project folder\n- Run: `pip install -r requirements.txt`\n- If that fails, try: `pip3 install -r requirements.txt`\n- Use a virtual environment: `python -m venv venv && source venv/bin/activate`\n- Check Python version first: `python --version` (need 3.10+)"
    },
    {
        "user": "My model keeps repeating itself.",
        "assistant": "- Lower the `frequency_penalty` parameter (try 0.5–1.0)\n- Reduce max_tokens if response is too long\n- Check your system prompt for conflicting instructions\n- Try a different temperature (0.7 is usually balanced)\n- Review the conversation history for repeated context"
    },
    {
        "user": "How do I update the app to the latest version?",
        "assistant": "- Stop the running container: `Ctrl+C` in the terminal\n- Pull the latest code: `git pull origin main`\n- Rebuild the Docker image: `docker build -t myapp .`\n- Restart: `docker run -p 8501:8501 myapp`\n- Clear browser cache if the UI looks outdated"
    },
    {
        "user": "The chatbot is ignoring my system prompt.",
        "assistant": "- Verify the system message is the first item in the `messages` array\n- Keep the system prompt concise and unambiguous\n- Avoid conflicting instructions between system and user messages\n- Try `gpt-4o` instead of older models for better instruction-following\n- Test with a minimal prompt to isolate the issue"
    },
    {
        "user": "How do I export my results?",
        "assistant": "- Look for a 'Download' or 'Export' button in the app sidebar\n- Results are usually available as CSV or JSON\n- Use `pd.DataFrame.to_csv('output.csv')` in code\n- Check the browser's download folder after export\n- For large exports, run the script directly in a terminal"
    },
    {
        "user": "I see ModuleNotFoundError in my terminal.",
        "assistant": "- The required library is not installed\n- Run: `pip install <module_name>`\n- Check requirements.txt — the module may be listed under a different package name\n- Make sure you're in the correct virtual environment\n- Restart the app after installing"
    },
    {
        "user": "Can the app run without internet?",
        "assistant": "- Modules using OpenAI API require internet\n- Modules using Ollama can run fully offline\n- Download the Ollama model first while online: `ollama pull llama3.2`\n- After download, disconnect and Ollama continues to work\n- The MCP Explorer and Cost Calculator also work offline"
    },
    {
        "user": "How do I add my own documents to the RAG demo?",
        "assistant": "- Open the RAG Chat page in the sidebar\n- Look for the file uploader widget\n- Upload a PDF or TXT file\n- Click 'Index' or 'Process' to chunk and embed the document\n- Your document is now searchable in the chat"
    }
]

# Convert to JSONL format
jsonl_lines = []
for ex in training_examples:
    record = {
        "messages": [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": ex["user"]},
            {"role": "assistant", "content": ex["assistant"]}
        ]
    }
    jsonl_lines.append(json.dumps(record))

# Write to file
with open("training_data.jsonl", "w") as f:
    f.write("\n".join(jsonl_lines))

print(f"✅ Created training_data.jsonl with {len(jsonl_lines)} examples")
print("\nSample record:")
print(json.dumps(json.loads(jsonl_lines[0]), indent=2))

## Part 3: Validate the Training Data

Before uploading, always validate your data. Common mistakes:
- Missing `assistant` messages
- Messages that are too short (< 5 words)
- Inconsistent formatting
- Duplicate examples

In [ ]:
# Validate the training data
errors = []
warnings = []
token_counts = []

with open("training_data.jsonl", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    try:
        record = json.loads(line.strip())
        messages = record.get("messages", [])
        
        roles = [m["role"] for m in messages]
        
        if "system" not in roles:
            errors.append(f"Line {i+1}: Missing system message")
        if "user" not in roles:
            errors.append(f"Line {i+1}: Missing user message")
        if "assistant" not in roles:
            errors.append(f"Line {i+1}: Missing assistant message")
        
        # Rough token estimate (1 token ≈ 4 chars)
        total_chars = sum(len(m["content"]) for m in messages)
        est_tokens = total_chars // 4
        token_counts.append(est_tokens)
        
        if est_tokens < 20:
            warnings.append(f"Line {i+1}: Very short example (~{est_tokens} tokens)")
        if est_tokens > 4096:
            errors.append(f"Line {i+1}: Example too long (~{est_tokens} tokens, max 4096)")
            
    except json.JSONDecodeError as e:
        errors.append(f"Line {i+1}: JSON parse error — {e}")

print(f"📊 Validation Report")
print(f"   Examples:  {len(lines)}")
print(f"   Errors:    {len(errors)}")
print(f"   Warnings:  {len(warnings)}")
print(f"   Avg tokens per example: {sum(token_counts)//len(token_counts) if token_counts else 0}")

if errors:
    print("\n❌ ERRORS (must fix before uploading):")
    for e in errors:
        print(f"   {e}")
else:
    print("\n✅ No errors found — data is ready to upload")

if warnings:
    print("\n⚠️  Warnings:")
    for w in warnings:
        print(f"   {w}")

## Part 4: Upload Data and Start a Fine-Tuning Job

Fine-tuning happens in two steps:
1. **Upload** the training file to OpenAI's servers
2. **Start** a fine-tuning job, referencing that file

The job typically takes **5–30 minutes** depending on dataset size.

In [ ]:
# Step 1: Upload the training file
print("Uploading training data...")

with open("training_data.jsonl", "rb") as f:
    upload_response = client.files.create(
        file=f,
        purpose="fine-tune"
    )

file_id = upload_response.id
print(f"✅ File uploaded: {file_id}")
print(f"   Status: {upload_response.status}")

In [ ]:
# Step 2: Start the fine-tuning job
# We use gpt-4o-mini as the base model — it is the most cost-effective option
print("Starting fine-tuning job...")

ft_job = client.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini-2024-07-18",   # base model to fine-tune
    hyperparameters={
        "n_epochs": 3,                 # 3 passes through the training data
    },
    suffix="support-bullets"          # adds this suffix to the model name
)

job_id = ft_job.id
print(f"✅ Fine-tuning job created: {job_id}")
print(f"   Status:     {ft_job.status}")
print(f"   Base model: {ft_job.model}")
print(f"   Training file: {ft_job.training_file}")
print()
print("⏳ The job will run for approximately 5–20 minutes.")
print("   Run the next cell to monitor progress.")

In [ ]:
# Monitor the fine-tuning job
# Run this cell repeatedly until status is 'succeeded'

job_status = client.fine_tuning.jobs.retrieve(job_id)

print(f"Job ID:       {job_status.id}")
print(f"Status:       {job_status.status}")
print(f"Created:      {job_status.created_at}")
if job_status.finished_at:
    print(f"Finished:     {job_status.finished_at}")
if job_status.fine_tuned_model:
    print(f"Fine-tuned model: {job_status.fine_tuned_model}")
    print()
    print("✅ Training complete! Copy the model name above for use in the next section.")
else:
    print()
    print("⏳ Still training... re-run this cell in a few minutes.")

# Show the last few training events
events = client.fine_tuning.jobs.list_events(job_id, limit=5)
print()
print("Recent events:")
for event in reversed(list(events)):
    print(f"  [{event.created_at}] {event.message}")

## Part 5: Use the Fine-Tuned Model

Once the job status is `succeeded`, you will have a new model name like:
`ft:gpt-4o-mini-2024-07-18:your-org:support-bullets:abc123`

You use it exactly like the base model — just reference it by name.

In [ ]:
# After training is complete, set your fine-tuned model name here
# Replace with the actual model name from the output above
FINE_TUNED_MODEL = "ft:gpt-4o-mini-2024-07-18:your-org:support-bullets:abc123"  # <-- replace this
BASE_MODEL       = "gpt-4o-mini"

def ask_model(model_name, question):
    """Ask a model a question and return the response."""
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system",  "content": SYSTEM_MSG},
            {"role": "user",    "content": question}
        ],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content

# Test questions — not in the training set
test_questions = [
    "The login page keeps showing a 403 error.",
    "How do I back up my conversation history?",
    "Can I use this app on my phone?"
]

print("=" * 70)
print("SIDE-BY-SIDE COMPARISON: Base Model vs. Fine-Tuned Model")
print("=" * 70)

for q in test_questions:
    print(f"\n❓ Question: {q}")
    print("-" * 70)
    
    base_answer = ask_model(BASE_MODEL, q)
    ft_answer   = ask_model(FINE_TUNED_MODEL, q)
    
    print(f"📦 Base ({BASE_MODEL}):")
    print(base_answer)
    print()
    print(f"🎯 Fine-tuned ({FINE_TUNED_MODEL[:35]}...):")
    print(ft_answer)
    print()

## Part 6: Cost and Tradeoffs

Fine-tuning is powerful but has real costs. Here is the decision framework:

| Approach | Setup Time | Cost | Best For |
|---|---|---|---|
| Prompt Engineering | Minutes | Free | Quick iteration, style guidance |
| RAG | Hours | Low | Grounding answers in documents |
| Fine-Tuning | Days | Medium | Consistent tone, format, domain style |

### OpenAI Fine-Tuning Costs (as of mid-2024)

| Model | Training | Inference Input | Inference Output |
|---|---|---|---|
| gpt-4o-mini | $0.003 / 1K tokens | $0.0003 / 1K | $0.0012 / 1K |
| gpt-3.5-turbo | $0.008 / 1K tokens | $0.003 / 1K | $0.006 / 1K |

### When Fine-Tuning Is Worth It

✅ **Good use cases:**
- Consistent output format (always bullet points, always JSON, always a specific structure)
- Brand voice and tone that prompt engineering can't reliably produce
- Reducing system prompt length (bake instructions into the model)
- High-volume production systems where latency and cost matter

❌ **Poor use cases:**
- Teaching the model new factual knowledge (use RAG instead)
- One-off tasks (the cost of training isn't justified)
- Rapidly changing requirements (fine-tuning is slow to iterate)

In [ ]:
# Cost Calculator: Estimate your fine-tuning job cost

# Training data stats
total_training_tokens = sum(
    sum(len(m["content"]) // 4 for m in json.loads(line)["messages"])
    for line in open("training_data.jsonl")
)
n_epochs = 3
total_billed_tokens = total_training_tokens * n_epochs

# gpt-4o-mini fine-tuning price
training_cost_per_1k = 0.003
training_cost = (total_billed_tokens / 1000) * training_cost_per_1k

# Inference cost estimate (1000 calls @ 500 tokens each)
inference_calls = 1000
tokens_per_call  = 500
inference_input_cost  = (inference_calls * tokens_per_call / 1000) * 0.0003
inference_output_cost = (inference_calls * tokens_per_call / 1000) * 0.0012

print("💰 Fine-Tuning Cost Estimate")
print("=" * 40)
print(f"Training examples:      {12}")
print(f"Estimated tokens/file:  ~{total_training_tokens:,}")
print(f"Epochs:                 {n_epochs}")
print(f"Total billed tokens:    ~{total_billed_tokens:,}")
print(f"Training cost:          ${training_cost:.4f}")
print()
print(f"After training — inference cost for {inference_calls:,} calls:")
print(f"  Input cost:           ${inference_input_cost:.4f}")
print(f"  Output cost:          ${inference_output_cost:.4f}")
print(f"  Total inference:      ${inference_input_cost + inference_output_cost:.4f}")
print()
print(f"Total cost (training + 1K calls): ${training_cost + inference_input_cost + inference_output_cost:.4f}")

## Part 7: Clean Up

Fine-tuned models continue to incur storage costs. Delete models you no longer need.

In [ ]:
# List all your fine-tuned models
print("Your fine-tuned models:")
models = client.models.list()
ft_models = [m for m in models if m.id.startswith("ft:")]

if ft_models:
    for m in ft_models:
        print(f"  {m.id}")
else:
    print("  No fine-tuned models found (training may still be in progress)")

# To delete a model (uncomment and replace the model name):
# client.models.delete("ft:gpt-4o-mini-2024-07-18:your-org:support-bullets:abc123")
# print("Model deleted.")

## Summary

In this notebook you:

- **Built a training dataset** in the JSONL format OpenAI requires
- **Validated the data** programmatically before uploading
- **Uploaded and launched** a fine-tuning job via the API
- **Compared base vs. fine-tuned** model responses side-by-side
- **Estimated costs** and understood the decision framework

### Key Takeaways

1. Fine-tuning shapes **style and format**, not factual knowledge
2. Always **prompt engineer first** — fine-tune only when prompts hit a wall
3. **Data quality beats data quantity** — 50 great examples outperform 500 mediocre ones
4. Fine-tuning is a **production tool** — the overhead is justified at scale

---

## 🚀 What's Next?

You have now completed Part 2 of the GenAI Foundry curriculum!

**Part 3 — Bridge to Agentic AI:**
- [Notebook 12: What is an Agent?](what_is_an_agent.ipynb) — From chatbots to agents: the Observe→Think→Act loop
- Then continue to [AgenticAI Foundry](https://github.com/dlwhyte/AgenticAI_foundry) for multi-agent systems, MCP, and agent security